In [1]:
!pip install torch==2.6.0 torchvision torchaudio

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta

In [2]:
import torch
print(torch.__version__)

2.6.0+cu124


In [3]:
!pip install "docling[pdf]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.0/506.0 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 87.2 MB/s eta 0:00:00
   ━

In [4]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 68.5 MB/s eta 0:00:00


In [5]:
!pip install -U transformers

In [128]:
from docling.document_converter import DocumentConverter
from docling.datamodel.document import TableItem, SectionHeaderItem
from collections import Counter
import fitz

def load_pdf_hybrid(source, debug=False):
    """
    Hybrid PDF loader:
    - Docling  → structure, multi-column reading order, tables, page provenance
    - PyMuPDF  → fallback page detection only if Docling provenance is missing
    """

    # ── Step 1: PyMuPDF (fallback only) ──────────────────────────────────────
    pdf        = fitz.open(source)
    page_texts = {}
    for i in range(len(pdf)):
        page_texts[i + 1] = pdf[i].get_text()   # 1-based

    # ── Step 2: Docling conversion ────────────────────────────────────────────
    converter = DocumentConverter()
    result    = converter.convert(source)
    doc       = result.document

    blocks       = []
    total_blocks = 0
    filename     = source.split("/")[-1]

    # ── Step 3: Iterate Docling items ─────────────────────────────────────────
    for item, level in doc.iterate_items():

        # ── 3a. Resolve page number ───────────────────────────────────────────
        prov     = getattr(item, "prov", None)
        page_num = prov[0].page_no if prov else None

        # Fallback: word-overlap against PyMuPDF (only if no provenance)
        if page_num is None:
            raw = (
                getattr(item, "text", None)
                or getattr(item, "content", None)
                or ""
            ).strip()
            if raw:
                words       = set(raw.split())
                best_page   = -1
                max_overlap = 0
                for pg, pg_text in page_texts.items():
                    overlap = len(words & set(pg_text.split()))
                    if overlap > max_overlap:
                        max_overlap = overlap
                        best_page   = pg
                page_num = best_page if max_overlap > 0 else -1
            else:
                page_num = -1

        # ── 3b. Resolve text ──────────────────────────────────────────────────
        if isinstance(item, TableItem):
            # Tables: export as markdown — preserves rows, cols, headers
            try:
                text = item.export_to_markdown()
            except Exception:
                try:
                    text = item.export_to_dataframe().to_markdown(index=False)
                except Exception:
                    text = ""
        else:
            text = (
                getattr(item, "text", None)
                or getattr(item, "content", None)
                or ""
            ).strip()

        if not text:
            continue

        total_blocks += 1

        # ── 3c. Build block — your original format ────────────────────────────
        block_id = f"{filename}_{page_num}_{total_blocks}"

        metadata = {
            "source":       filename,
            "page":         page_num,
            "block_index":  total_blocks,
            "total_blocks": total_blocks,
        }

        blocks.append({
            "id":       block_id,
            "content":  text,
            "metadata": metadata,
            "length":   len(text),
            "level":    level
        })

    # ── Step 4: Debug output ──────────────────────────────────────────────────
    if debug:
        print(f"Total blocks : {len(blocks)}\n")

        print("── Sample blocks ──────────────────────────────────────────")
        for b in blocks[:5]:
            print(f"ID:    {b['id']}")
            print(f"Index: {b['metadata']['block_index']}")
            print(f"Level: {b['level']}")
            print(f"Len:   {b['length']} chars")
            print(f"Text:  {b['content'][:300]}")
            print("-" * 60)

        pg_dist = Counter(b["metadata"]["page"] for b in blocks)
        print(f"\n📊 Page distribution: {dict(pg_dist)}")

    return blocks


# ── Usage ─────────────────────────────────────────────────────────────────────
pdf_path = "/content/Bookshelf_NBK568220.pdf"
blocks   = load_pdf_hybrid(pdf_path, debug=True)
print(f"\n✅ Total blocks: {len(blocks)}")

[INFO] 2026-05-09 16:14:06,301 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-09 16:14:06,305 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-09 16:14:06,475 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-09 16:14:06,478 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-09 16:14:06,985 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-09 16:14:06,987 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-09 16:14:06,993 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-09 16:14:06,995 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Total blocks : 664

── Sample blocks ──────────────────────────────────────────
ID:    Bookshelf_NBK568220.pdf_1_1
Index: 1
Level: 1
Len:   21 chars
Text:  THE SURGEON GENERAL'S
------------------------------------------------------------
ID:    Bookshelf_NBK568220.pdf_1_2
Index: 2
Level: 1
Len:   14 chars
Text:  CALL TO ACTION
------------------------------------------------------------
ID:    Bookshelf_NBK568220.pdf_1_3
Index: 3
Level: 1
Len:   26 chars
Text:  TO IMPROVE MATERNAL HEALTH
------------------------------------------------------------
ID:    Bookshelf_NBK568220.pdf_1_4
Index: 4
Level: 1
Len:   42 chars
Text:  U.S. Department of Health & Human Services
------------------------------------------------------------
ID:    Bookshelf_NBK568220.pdf_2_5
Index: 5
Level: 1
Len:   73 chars
Text:  FOREWORD FROM THE SECRETARY, U.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES
------------------------------------------------------------

📊 Page distribution: {1: 4, 2: 12, 3: 9, 4: 3, 5: 2, 

In [35]:
def check_tables(blocks):
    tables = [b for b in blocks if b["content"].strip().startswith("|")]

    print(f"Total tables found: {len(tables)}")
    print()

    for i, b in enumerate(tables):
        print(f"── Table {i+1} | Page {b['metadata']['page']} | {b['length']} chars ──────────────────")
        print(b["content"])   # remove the [:500] limit
        print()

check_tables(blocks)

Total tables found: 6

── Table 1 | Page 4 | 6265 chars ──────────────────
| Table 1. Levels of Maternal Care: Definitions, Capabilities, and Health Care Providers*   | Table 1. Levels of Maternal Care: Definitions, Capabilities, and Health Care Providers*                                                                                                                                                                                                                                          |
|-------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Accredited Birth Center                                                                  

In [36]:
def show_blocks_by_page(blocks, page_num):
    """
    Display all blocks from a specific page number.

    Args:
        blocks: list of block dicts from load_pdf_hybrid()
        page_num: the page number to filter by (1-based)
    """
    page_blocks = [b for b in blocks if b["metadata"]["page"] == page_num]

    print(f"Total blocks on page {page_num}: {len(page_blocks)}")
    print("=" * 60)

    for b in page_blocks:
        print(f"ID:    {b['id']}")
        print(f"Index: {b['metadata']['block_index']}")
        print(f"Level: {b['level']}")
        print(f"Len:   {b['length']} chars")
        print(f"Text:  {b['content'][:300]}")
        print("-" * 60)

In [37]:
show_blocks_by_page(blocks,9)

Total blocks on page 9: 13
ID:    Levels of Maternal Care_ACOG.pdf_9_54
Index: 54
Level: 1
Len:   97 chars
Text:  established. Pregnant women should receive the same level of trauma care as nonpregnant patients.
------------------------------------------------------------
ID:    Levels of Maternal Care_ACOG.pdf_9_55
Index: 55
Level: 2
Len:   137 chars
Text:  c The appropriate care level for patients should be driven by their medical need and not limited to or governed by financial constraints.
------------------------------------------------------------
ID:    Levels of Maternal Care_ACOG.pdf_9_56
Index: 56
Level: 2
Len:   558 chars
Text:  c Because obesity is extremely common throughout the United States, all facilities should have appropriate equipment for the care and delivery of pregnant women with obesity, including appropriate birth beds, operating tables and rooms, and operating equipment (34). The degree of obesity may be one 
---------------------------------------------------

In [129]:
import re

# --------------------------------------------------
# 🔹 Sections to skip entirely
# --------------------------------------------------
SKIP_SECTIONS = {
    "acknowledgements", "acknowledgments", "foreword", "preface",
    "abbreviations", "acronyms", "bibliography", "references",
    "about the authors", "about the editors", "colophon",
    "conflict of interest", "funding", "appendix","endnotes"
}

# --------------------------------------------------
# 🔹 Detect index / table of contents pages
# --------------------------------------------------
def is_index_page(text):
    lines = text.split("\n")

    dot_lines = sum(1 for l in lines if re.search(r'\.{3,}\s*\d+', l))
    numbered_lines = sum(1 for l in lines if re.match(r'^\s*\d+[\.\)]?\s+', l))
    has_contents = re.search(r'\b(contents|table of contents)\b', text, re.IGNORECASE)

    if has_contents:
        return True
    if dot_lines > 3:
        return True
    if numbered_lines > 5 and len(text) < 2000:
        return True

    return False


# --------------------------------------------------
# 🔹 Detect skip-section headers
# --------------------------------------------------
def is_skip_section_header(text):
    cleaned = text.strip().lower()
    return cleaned in SKIP_SECTIONS or any(cleaned.startswith(s) for s in SKIP_SECTIONS)

# --------------------------------------------------
# 🔹 Detect endnote / reference blocks
# --------------------------------------------------
def is_reference_block(text):
    text = text.strip()

    # Starts with a number (endnote marker)
    if re.match(r'^\d+[\.\)]?\s+[A-Z]', text):
        return True

    # Contains journal citation patterns
    if re.search(r'\d{4};\d+\(\d+\):\d+', text):  # e.g. 2019;68(18):423-429
        return True

    # Contains DOI or URL remnants
    if re.search(r'(doi:|Published \d{4}|Accessed \w+ \d+)', text, re.IGNORECASE):
        return True

    # Author list pattern: LastName XX, LastName XX, et al.
    if re.search(r'[A-Z][a-z]+\s+[A-Z]{2},', text):
        return True

    return False
# --------------------------------------------------
# 🔹 Text cleaning
# --------------------------------------------------
def preprocess_text(text):

    # 1. Fix hyphenated line breaks
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)

    # 2. Remove citations
    text = re.sub(r'\[\d+([,\-]\d+)*\]', '', text)
    text = re.sub(r'\(\s*[A-Za-z][^)]*\d{4}[a-z]?\s*\)', '', text)

    # 3. Remove references / URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\bWebsite\b', '', text, flags=re.IGNORECASE)

    text = re.sub(
        r'^[A-Z][^.]+\.\s+[A-Z][^.]+\.\s+\w+:.*?\d{4}.*$',
        '',
        text,
        flags=re.MULTILINE
    )

    # 4. Remove standalone page numbers
    text = re.sub(r'^\s*[-–—]*\s*\d+\s*[-–—]*\s*$', '', text, flags=re.MULTILINE)

    # 5. Remove bullets/symbols
    text = re.sub(r'[•●■◆▪►▸→✓✗]', '', text)

    # 6. Remove headers/footers
    text = re.sub(r'WHO recommendations.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\s*(chapter|section|page)\s*\d+.*$', '', text,
                  flags=re.MULTILINE | re.IGNORECASE)

    # 7. Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    return text


# --------------------------------------------------
# 🔹 Block-level processing
# --------------------------------------------------
def preprocess_blocks(blocks, min_length=30):

    cleaned = []
    skip_mode = False  # 🔥 True when inside a skip section

    for block in blocks:
        raw_text = block.get("content", "")
        text = preprocess_text(raw_text)

        # ❌ Remove index / TOC pages
        if is_index_page(text):
            skip_mode = False
            continue

        # 🔥 If this block is a skip-section header, enter skip mode
        if is_skip_section_header(text):
            skip_mode = True
            continue

        # ✅ New major section header found — exit skip mode
        if skip_mode and block.get("level") in (1, 2):
            skip_mode = False

        if skip_mode:
            continue
        if is_reference_block(text):
            continue
        # Drop short noise
        if len(text) < min_length:
            continue

        # Drop pure numbers
        if re.match(r'^\d+$', text):
            continue

        # Drop low alphabet ratio (noise)
        alpha_ratio = sum(c.isalpha() for c in text) / max(len(text), 1)
        if alpha_ratio < 0.4:
            continue

        # ✅ Keep metadata intact (important!)
        cleaned.append({
            **block,
            "content": text,
            "length": len(text)
        })

    return cleaned

In [130]:
import re

# --------------------------------------------------
# 🔹 Sections to skip entirely
# --------------------------------------------------
SKIP_SECTIONS = {
    "acknowledgements", "acknowledgments", "foreword", "preface",
    "abbreviations", "acronyms", "bibliography", "references",
    "about the authors", "about the editors", "colophon",
    "conflict of interest", "funding", "appendix", "endnotes"
}

# --------------------------------------------------
# 🔹 Detect index / table of contents pages
# --------------------------------------------------
def is_index_page(text):
    lines = text.split("\n")
    dot_lines      = sum(1 for l in lines if re.search(r'\.{3,}\s*\d+', l))
    numbered_lines = sum(1 for l in lines if re.match(r'^\s*\d+[\.\)]?\s+', l))
    has_contents   = re.search(r'\b(contents|table of contents)\b', text, re.IGNORECASE)

    if has_contents:
        return True
    if dot_lines > 3:
        return True
    if numbered_lines > 5 and len(text) < 2000:
        return True
    return False


# --------------------------------------------------
# 🔹 Detect skip-section headers
# --------------------------------------------------
def is_skip_section_header(text):
    cleaned = text.strip().lower()
    return cleaned in SKIP_SECTIONS or any(cleaned.startswith(s) for s in SKIP_SECTIONS)


# --------------------------------------------------
# 🔹 Detect endnote / reference blocks (run on RAW text)
# --------------------------------------------------
def is_reference_block(text):
    t = text.strip()

    # Footnote symbol lines: †, ‡, §, *, ¶
    if re.match(r'^[†‡§*¶]', t):
        return True

    # Starts with a number (endnote marker)
    if re.match(r'^\d+[\.\)]?\s+[A-Z]', t):
        return True

    # Journal citation: 2019;68(18):423-429
    if re.search(r'\d{4};\d+\(\d+\):\d+', t):
        return True

    # Page range like e81-94 or 746-50 (journal page patterns)
    if re.search(r'\d{4};\d+.*:\w*\d+-\d+', t):
        return True

    # "Committee Opinion No. XXX" or "Practice Bulletin No. XX"
    if re.search(r'(Committee Opinion|Practice Bulletin|Practice Advisory)\s+No\.', t, re.IGNORECASE):
        return True

    # "Obstet Gynecol", "MMWR", "JAMA" etc — journal name signals
    if re.search(r'\b(Obstet Gynecol|MMWR|JAMA|Lancet|N Engl J Med|Am J Obstet)\b', t):
        return True

    # "Available at: Retrieved" or "Washington, DC:" publisher patterns
    if re.search(r'(Available at:|Retrieved \w+ \d+|Washington,\s*DC:)', t, re.IGNORECASE):
        return True

    # DOI or access/publish date
    if re.search(r'(doi:|Published \d{4}|Accessed \w+ \d+)', t, re.IGNORECASE):
        return True

    # URL present
    if re.search(r'https?://', t):
        return True

    # Author list: LastName XX, LastName XX
    if re.search(r'[A-Z][a-z]+\s+[A-Z]{2},', t):
        return True

    return False


# --------------------------------------------------
# 🔹 Text cleaning
# --------------------------------------------------
def preprocess_text(text):

    # 1. Fix hyphenated line breaks
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)

    # 2. Remove inline citations
    text = re.sub(r'\[\d+([,\-]\d+)*\]', '', text)
    text = re.sub(r'\(\s*[A-Za-z][^)]*\d{4}[a-z]?\s*\)', '', text)

    # 3. Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\bWebsite\b', '', text, flags=re.IGNORECASE)

    # 4. Remove standalone page numbers
    text = re.sub(r'^\s*[-–—]*\s*\d+\s*[-–—]*\s*$', '', text, flags=re.MULTILINE)

    # 5. Remove bullets/symbols
    text = re.sub(r'[•●■◆▪►▸→✓✗]', '', text)

    # 6. Remove repeated headers/footers
    text = re.sub(r'WHO recommendations.*?\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\s*(chapter|section|page)\s*\d+.*$', '', text,
                  flags=re.MULTILINE | re.IGNORECASE)

    # 7. Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()

    return text


# --------------------------------------------------
# 🔹 Block-level processing
# --------------------------------------------------
def preprocess_blocks(blocks, min_length=30):

    cleaned   = []
    skip_mode = False

    for block in blocks:
        raw_text = block.get("content", "")

        # ✅ Check references on RAW text BEFORE any cleaning
        if is_reference_block(raw_text):
            continue

        text = preprocess_text(raw_text)

        # Remove index / TOC pages
        if is_index_page(text):
            skip_mode = False
            continue

        # Enter skip mode on section headers like "References", "Appendix"
        if is_skip_section_header(text):
            skip_mode = True
            continue

        # Exit skip mode when a new real section starts
        if skip_mode and block.get("level") in (1, 2):
            skip_mode = False

        if skip_mode:
            continue

        # Drop short noise
        if len(text) < min_length:
            continue

        # Drop pure numbers
        if re.match(r'^\d+$', text):
            continue

        # Drop low alphabet ratio (noise / symbols)
        alpha_ratio = sum(c.isalpha() for c in text) / max(len(text), 1)
        if alpha_ratio < 0.4:
            continue

        cleaned.append({
            **block,
            "content": text,
            "length":  len(text)
        })

    return cleaned

In [131]:
cleaned_blocks = preprocess_blocks(blocks, min_length=30)

print(f"✅ Before: {len(blocks)} blocks")
print(f"✅ After:  {len(cleaned_blocks)} blocks")
print(f"🗑️  Removed: {len(blocks) - len(cleaned_blocks)} noisy blocks")

print("\n📌 Sample cleaned blocks:")

for b in cleaned_blocks[:3]:
    print(f"\nID: {b['id']}")
    print(f"Page: {b['metadata']['page']}")
    print(f"Level: {b.get('level', 'N/A')}")
    print(f"Length: {b['length']}")
    print(f"Text: {b['content'][:200]}")
    print("-" * 60)

✅ Before: 664 blocks
✅ After:  338 blocks
🗑️  Removed: 326 noisy blocks

📌 Sample cleaned blocks:

ID: Bookshelf_NBK568220.pdf_1_4
Page: 1
Level: 1
Length: 42
Text: U.S. Department of Health & Human Services
------------------------------------------------------------

ID: Bookshelf_NBK568220.pdf_2_6
Page: 2
Level: 1
Length: 638
Text: The United States has one of the most technologically advanced healthcare systems in the world, yet we have a maternal mortality rate that is higher than comparable countries. Racial and ethnic, geogr
------------------------------------------------------------

ID: Bookshelf_NBK568220.pdf_2_7
Page: 2
Level: 1
Length: 415
Text: The disparities go well beyond tragic, unnecessary deaths. Each year, thousands of women experience severe maternal morbidity-unexpected outcomes of labor and delivery that result in significant short
------------------------------------------------------------


In [13]:
!pip install langchain-text-splitters -q

In [132]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")


# --------------------------------------------------
# Sentence splitting (spaCy)
# --------------------------------------------------
def sentence_split(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


# --------------------------------------------------
# Pack sentences into semantic chunks
# --------------------------------------------------
def pack_sentences(sentences, max_chars=800, overlap_sents=1):
    chunks        = []
    current_sents = []
    current_len   = 0

    for sent in sentences:
        if current_len + len(sent) > max_chars and current_sents:
            chunks.append(" ".join(current_sents).strip())

            # overlap: carry last N full sentences — never cuts mid-sentence
            current_sents = current_sents[-overlap_sents:] if overlap_sents else []
            current_len   = sum(len(s) for s in current_sents)

        current_sents.append(sent)
        current_len += len(sent)

    if current_sents:
        chunks.append(" ".join(current_sents).strip())

    return chunks


# --------------------------------------------------
# Table detection (structure-based)
# --------------------------------------------------
def is_table_like(text):
    lines = [l for l in text.split("\n") if l.strip()]

    if len(lines) < 3:
        return False

    pipe_lines        = sum(1 for l in lines if "|" in l)
    multi_space_lines = sum(1 for l in lines if re.search(r"\s{3,}", l))
    numeric_lines     = sum(1 for l in lines if sum(c.isdigit() for c in l) > 3)

    score = pipe_lines + multi_space_lines + numeric_lines

    return score >= 3


# --------------------------------------------------
# Table row-based chunking (header repeated each chunk)
# --------------------------------------------------
def chunk_table_rows(text, current_header, current_level, page):
    lines = [l for l in text.split("\n") if l.strip()]
    pipe_lines = [l for l in lines if "|" in l]

    # not enough structure — keep as-is
    if len(pipe_lines) < 3:
        return [{
            "text":   "TABLE:\n" + "\n".join(lines),
            "header": current_header,
            "level":  current_level,
            "source": "table",
            "page":   page
        }]

    header    = pipe_lines[0]
    data_rows = [l for l in pipe_lines[1:]
                 if not all(c in "|-: " for c in l)]  # skip separator line

    chunks = []
    for i, row in enumerate(data_rows):
        chunks.append({
            "text":   f"TABLE: {current_header}\n{header}\n{row}",
            "header": current_header,
            "level":  current_level,
            "source": f"table_row_{i+1}",
            "page":   page
        })

    return chunks


# --------------------------------------------------
# SMART CHUNKING (FINAL FIXED VERSION)
# --------------------------------------------------
def smart_chunk_blocks(cleaned_blocks,
                       chunk_size=800,
                       chunk_overlap=100,
                       small_block_threshold=300):

    chunks = []

    current_header = "Introduction"
    current_level  = 1

    for block in cleaned_blocks:

        text = (block.get("content") or "").strip()
        if not text:
            continue

        level = block.get("level", 1)
        page  = block.get("metadata", {}).get("page", -1)

        # --------------------------------------------------
        # HEADER TRACKING
        # --------------------------------------------------
        if len(text) < 120 and level in [1, 2]:
            current_header = text
            current_level  = level
            continue

        # --------------------------------------------------
        # TABLE HANDLING
        # --------------------------------------------------
        if is_table_like(text):
            chunks.extend(chunk_table_rows(
                text, current_header, current_level, page
            ))
            continue

        # --------------------------------------------------
        # SMALL BLOCKS
        # --------------------------------------------------
        if len(text) <= small_block_threshold:
            chunks.append({
                "text":   text,
                "header": current_header,
                "level":  current_level,
                "source": "small_block",
                "page":   page
            })
            continue

        # --------------------------------------------------
        # LARGE BLOCK → semantic split
        # --------------------------------------------------
        sentences = sentence_split(text)

        packed_chunks = pack_sentences(
            sentences,
            max_chars=chunk_size,
            overlap_sents=1
        )

        for i, chunk in enumerate(packed_chunks):
            chunks.append({
                "text":   chunk,
                "header": current_header,
                "level":  current_level,
                "source": f"semantic_split_{i+1}",
                "page":   page
            })

    return chunks

In [133]:
chunks = smart_chunk_blocks(
    cleaned_blocks,
    chunk_size=400,
    chunk_overlap=80
)

print(f"✅ Total chunks: {len(chunks)}")

✅ Total chunks: 428


In [43]:
import statistics
from collections import Counter

# ── Size distribution ─────────────────────────────────────
lengths = [len(c["text"]) for c in chunks]

print(f"\n📏 Chunk size stats:")
print(f"  Min    : {min(lengths)}")
print(f"  Max    : {max(lengths)}")
print(f"  Mean   : {statistics.mean(lengths):.1f}")
print(f"  Median : {statistics.median(lengths):.1f}")


# ── Source breakdown ──────────────────────────────────────
sources = Counter(c["source"] for c in chunks)

print(f"\n📊 Chunk sources:")
for s, count in sources.most_common():
    print(f"  {s}: {count}")


# ── Page distribution (NEW) ───────────────────────────────
pages = [c.get("page", -1) for c in chunks]
page_counts = Counter(pages)

print(f"\n📄 Page distribution:")
for p, count in page_counts.most_common(10):
    print(f"  Page {p}: {count} chunks")


# ── Preview chunks ────────────────────────────────────────
print("\n📌 Sample chunks:")

for c in chunks[5:8]:
    print(f"\n  Header : {c['header'][:60]}")
    print(f"  Page   : {c.get('page', -1)}")
    print(f"  Source : {c['source']}")
    print(f"  Text   : {c['text'][:200]}")
    print("  " + "-"*50)


📏 Chunk size stats:
  Min    : 121
  Max    : 1752
  Mean   : 421.2
  Median : 384.0

📊 Chunk sources:
  semantic_split_1: 36
  semantic_split_2: 31
  semantic_split_3: 19
  semantic_split_4: 14
  small_block: 13
  semantic_split_5: 10
  table_row_1: 6
  table_row_2: 6
  semantic_split_6: 5
  table_row_3: 4
  semantic_split_7: 3
  semantic_split_8: 3
  table_row_4: 3
  table_row_5: 3
  table_row_6: 2
  table_row_7: 2
  table_row_8: 1
  table_row_9: 1

📄 Page distribution:
  Page 2: 28 chunks
  Page 9: 23 chunks
  Page 3: 22 chunks
  Page 11: 22 chunks
  Page 1: 16 chunks
  Page 8: 12 chunks
  Page 15: 11 chunks
  Page 10: 10 chunks
  Page 4: 7 chunks
  Page 7: 4 chunks

📌 Sample chunks:

  Header : Introduction
  Page   : 1
  Source : semantic_split_6
  Text   : The determination of the appropriate level of care to be provided by a given facility should be guided by regional and state health care entities, national accreditation and professional organization 
  -----------------------

In [46]:
page_no = 14

page_chunks = [c for c in chunks if c.get("page") == page_no]

print(f"\n📄 Total chunks in page {page_no}: {len(page_chunks)}")

for i, c in enumerate(page_chunks[:10]):  # show first 10 only
    print(f"\n🔹 Chunk {i+1}")
    print(f"Header : {c['header']}")
    print(f"Source : {c['source']}")
    print(f"Text   : {c['text']}")
    print("-" * 60)


📄 Total chunks in page 14: 0


In [ ]:
page_no =

page_chunks = [c for c in chunks if c.get("page") == page_no]

c = page_chunks[]  # index 9 = chunk 10

print(f"Header : {c['header']}")
print(f"Source : {c['source']}")
print(f"Text   :\n{c['text']}")

Header : Pregestational diabetes mellitus. ACOG Practice Bulletin No. 201. American College of Obstetricians and Gynecologists.
Source : small_block
Text   :
National Academies of Sciences, Engineering, and Medicine. Integrating social care into the delivery of health care: moving upstream to improve the nation ' s health. National Academy of Sciences; 2019.


In [134]:
# Too tiny
tiny = [c for c in chunks if len(c["text"]) < 40]
print(f"\n⚠️  Tiny chunks (<40 chars): {len(tiny)}")
for t in tiny[:5]:
    print(f"   [{t['header'][:40]}] → '{t['text']}'")

# Too large (shouldn't happen but good to verify)
huge = [c for c in chunks if len(c["text"]) > 600]
print(f"\n⚠️  Huge chunks (>600 chars): {len(huge)}")


⚠️  Tiny chunks (<40 chars): 0

⚠️  Huge chunks (>600 chars): 20


In [135]:
import re

def filter_chunks(chunks, min_length=20):
    skip_patterns = [
        r'©\s*World Health Organization',
        r'All rights reserved',
        r'Licence:\s*CC',
        r'Suggested citation',
        r'WHO is not responsible',
    ]

    pattern = re.compile('|'.join(skip_patterns), re.IGNORECASE)

    filtered = []

    for c in chunks:
        text = c.get("text", "").strip()

        if not text:
            continue

        noise_score = 0

        # --------------------------------------------------
        # 1. length check (soft)
        # --------------------------------------------------
        if len(text) < min_length:
            noise_score += 1

        # --------------------------------------------------
        # 2. boilerplate detection
        # --------------------------------------------------
        if pattern.search(text):
            noise_score += 2

        # --------------------------------------------------
        # 3. URL presence (soft penalty, not removal)
        # --------------------------------------------------
        if re.search(r'http[s]?://', text):
            noise_score += 1

        # --------------------------------------------------
        # DECISION RULE (soft filter)
        # --------------------------------------------------
        if noise_score >= 3:
            continue  # only drop very noisy chunks

        # keep chunk but optionally tag quality
        c["noise_score"] = noise_score
        filtered.append(c)

    return filtered


# --------------------------------------------------
# APPLY FILTER
# --------------------------------------------------
chunks_clean = filter_chunks(chunks)

print(f"✅ Before filter: {len(chunks)}")
print(f"✅ After filter:  {len(chunks_clean)}")

✅ Before filter: 428
✅ After filter:  428


In [21]:
!pip install FlagEmbedding

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.2 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=fdfbcba5bac6099b712d8ba47799258e23e519c6805eee4b51c91b2c4faba88e
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Created wheel for cbor: filename=cbor-1.0.0-cp312-cp312-linux_x86_64.whl size=55022 sha256=9564fc65494193f5426d552ee1982bf2a60bf91cad8e27479d331dbc0d545ac1
 

In [136]:
import os
os.environ["TRANSFORMERS_SAFETENSORS_FAST"] = "1"
os.environ["USE_SAFETENSORS"] = "1"

from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

import numpy as np

# ── Load BEST multilingual model ─────────────────────────────
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)


def embed_chunks(chunks, batch_size=32):
    """
    Embeds English + Bangla chunks using BGE-M3.
    Produces dense vectors for Pinecone.
    """

    texts = []

    for c in chunks:
        text = c.get("text", "")
        header = c.get("header", "")

        # context-aware embedding (VERY IMPORTANT)
        combined = f"Section: {header}\n\n{text}"
        texts.append(combined)

    print(f"🔄 Embedding {len(texts)} chunks...")

    outputs = model.encode(
        texts,
        batch_size=batch_size,
        max_length=1024,
        return_dense=True
    )

    embeddings = outputs["dense_vecs"]

    embedded_chunks = []

    for i, c in enumerate(chunks):
        embedded_chunks.append({
            **c,
            "embedding": embeddings[i]
        })

    return embedded_chunks, embeddings

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [137]:
embedded_chunks, embeddings = embed_chunks(chunks_clean)

🔄 Embedding 428 chunks...


Inference Embeddings: 100%|██████████| 14/14 [00:02<00:00,  5.21it/s]


In [138]:
import hashlib
import os

def build_metadata_chunks(chunks,
                           pdf_path,
                           source_org="U.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES",
                           pub_year=2020,
                           doc_id="SURGEON_GENERAL’S_CALL_TO_ACTION",
                           authority_level="secondary"):

    filename = os.path.basename(pdf_path)
    total = len(chunks)

    structured = []

    for i, chunk in enumerate(chunks):

        text = chunk.get("text", "")

        if not text:
            continue

        # ─────────────────────────────
        # Unique ID
        # ─────────────────────────────
        uid = hashlib.md5(
            f"{doc_id}_{i}_{text[:50]}".encode()
        ).hexdigest()[:12]

        # ─────────────────────────────
        # USE EXISTING EXTRACTED DATA
        # ─────────────────────────────
        page = chunk.get("page", -1)
        header = chunk.get("header", "")
        chunk_type = chunk.get("type", "")
        chunk_source = chunk.get("source", "")

        # ─────────────────────────────
        # FINAL STRUCTURE (PINECONE READY)
        # ─────────────────────────────
        record = {
            "id": uid,
            "content": text,
            "embedding": chunk.get("embedding"),

            "metadata": {
                "source": filename,
                "page": page,
                "chunk_index": i,
                "total_chunks": total,
                "source_org": source_org,
                "pub_year": pub_year,
                "doc_id": doc_id,
                "authority_level": authority_level,

                # extracted (NOT manually invented)
                "section_title": header,
                "block_type": chunk_type,
                "chunk_source": chunk_source
            }
        }

        structured.append(record)

    return structured

In [139]:
structured_chunks = build_metadata_chunks(
    embedded_chunks,   # ✅ FIXED VARIABLE NAME
    pdf_path=pdf_path,
    source_org="U.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES",
    pub_year=2020,
    doc_id="SURGEON_GENERAL’S_CALL_TO_ACTION",
    authority_level="secondary"
)

print(f"✅ Total structured chunks: {len(structured_chunks)}")

✅ Total structured chunks: 428


In [140]:
# Preview one
import json
sample = structured_chunks[15].copy()
sample["embedding"] = f"[vector dim {len(sample['embedding'])}]"  # don't print full vector
print(json.dumps(sample, indent=2))

{
  "id": "e172db41f203",
  "content": "The death of a woman from pregnancy-related causes is one of the greatest tragedies that can befall a family and a community. Sadly, this catastrophe happens about 700 times each year in the U.S. -- far more frequently than in countries of similar population size and income.",
  "embedding": "[vector dim 1024]",
  "metadata": {
    "source": "Bookshelf_NBK568220.pdf",
    "page": 3,
    "chunk_index": 15,
    "total_chunks": 428,
    "source_org": "U.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES",
    "pub_year": 2020,
    "doc_id": "SURGEON_GENERAL\u2019S_CALL_TO_ACTION",
    "authority_level": "secondary",
    "section_title": "Alex M. Azar II Secretary of Health and Human Services",
    "block_type": "",
    "chunk_source": "semantic_split_2"
  }
}


In [54]:
import numpy as np
from FlagEmbedding import BGEM3FlagModel

# ── Load same model used for embeddings ─────────────
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)


# ── Encode query ────────────────────────────────────
def encode_query(query):
    outputs = model.encode(
        [query],
        return_dense=True
    )
    return outputs["dense_vecs"][0]


# ── Similarity search ───────────────────────────────
def search(query, embedded_chunks, top_k=5):

    query_vec = encode_query(query)

    scores = []

    for chunk in embedded_chunks:
        chunk_vec = chunk["embedding"]

        # cosine similarity (manual, no sklearn needed)
        score = np.dot(query_vec, chunk_vec) / (
            np.linalg.norm(query_vec) * np.linalg.norm(chunk_vec)
        )

        scores.append((score, chunk))

    # sort best matches first
    scores.sort(reverse=True, key=lambda x: x[0])

    return scores[:top_k]


# ── TEST QUERY ──────────────────────────────────────
query = "stigma for obesity"

results = search(query, embedded_chunks, top_k=5)

# ── PRINT RESULTS ───────────────────────────────────
print("\n🔎 TOP RESULTS:\n")

for score, chunk in results:
    print(f"Score: {score:.4f}")
    print(f"Page : {chunk.get('page', 'N/A')}")
    print(f"Header: {chunk.get('header', '')}")
    print(f"Text : {chunk['text'][:500]}")
    print("-" * 80)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


🔎 TOP RESULTS:

Score: 0.4939
Page : 9
Header: established. Pregnant women should receive the same level of trauma care as nonpregnant patients.
Text : c Because obesity is extremely common throughout the United States, all facilities should have appropriate equipment for the care and delivery of pregnant women with obesity, including appropriate birth beds, operating tables and rooms, and operating equipment (34). The degree of obesity may be one of the factors that affects decisions for transfer of a woman to a higher level of care, although there are no well-established body mass index cut-off levels to determine level-specific care for preg
--------------------------------------------------------------------------------
Score: 0.4893
Page : 9
Header: established. Pregnant women should receive the same level of trauma care as nonpregnant patients.
Text : c Because obesity is extremely common throughout the United States, all facilities should have appropriate equipment for the care

In [55]:
# ─────────────────────────────────────────────
# 🔎 QUERY TESTING (ENGLISH + BANGLA)
# ─────────────────────────────────────────────

queries = [
    # English queries
    "neonatal morbidity",
    "goals of regionalized maternal care",

   # Bangla queries
"টেলিমেডিসিন কী",
"মাতৃ ঝুঁকি মূল্যায়নের চিকিৎসাগত উপাদানসমূহ কী কী"
]

for q in queries:
    print("\n" + "="*90)
    print(f"🔍 QUERY: {q}")

    results = search(q, embedded_chunks, top_k=3)

    for score, chunk in results:
        print(f"\nScore : {score:.4f}")
        print(f"Page  : {chunk.get('page', 'N/A')}")
        print(f"Header: {chunk.get('header', '')}")
        print(f"Text  : {chunk['text'][:200]}")
        print("-"*70)


🔍 QUERY: neonatal morbidity

Score : 0.6118
Page  : 2
Header: Introduction
Text  : A comprehensive meta-analysis showed an increased risk of neonatal mortality for very-low-birth-weight infants (less than 1,500 g) born outside of a neonatal intensive care unit level III hospital (38
----------------------------------------------------------------------

Score : 0.6113
Page  : 2
Header: Introduction
Text  : Numerous studies validated the concept that improved neonatal outcomes were achieved through the application of risk-appropriate maternal transport systems (11, 12). A comprehensive meta-analysis show
----------------------------------------------------------------------

Score : 0.6035
Page  : 2
Header: Introduction
Text  : Similarly, neonatal mortality was higher for very-low-birth-weight infants born in hospitals staffed by neonatologists in the absence of a more complete multidisciplinary team (level II), com- pared w
-------------------------------------------------------------

In [29]:
# NEW CELL 1: Install Pinecone
!pip install pinecone sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 26.2 MB/s eta 0:00:00


In [58]:
from pinecone import Pinecone, ServerlessSpec

# Connect Pinecone
pc = Pinecone(api_key="pcsk_2X2Gyr_De54caT3PgyLKzBKURoA8SyRA8zA3fFkbjMA5fqZ5qkcCkBgZ3TPKVjXmJNjwGn")

# Main index name
index_name = "maternal-health"

# Create only once
if index_name not in pc.list_indexes().names():

    pc.create_index(
        name=index_name,
        dimension=1024,   # BGE-M3 dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("✅ Main index ready")

✅ Main index ready


In [144]:
# Connect to existing index
index = pc.Index("maternal-health")

# Namespace for THIS PDF
namespace = "SURGEON_GENERAL_CALL_TO_ACTION"

print("✅ Namespace selected:", namespace)

✅ Namespace selected: SURGEON_GENERAL_CALL_TO_ACTION


In [145]:
import numpy as np

vectors = []

for c in structured_chunks:

    embedding = np.array(c["embedding"]).astype(float).tolist()

    metadata = c["metadata"].copy()

    # Store actual chunk text
    metadata["content"] = c["content"]

    vectors.append({
        "id": c["id"],
        "values": embedding,
        "metadata": metadata
    })

print("✅ Vectors prepared:", len(vectors))



✅ Vectors prepared: 428


In [146]:
batch_size = 100

for i in range(0, len(vectors), batch_size):

    index.upsert(
        vectors=vectors[i:i+batch_size],
        namespace=namespace
    )

    print(f"✅ Uploaded batch {i//batch_size + 1}")

print("✅ Upload complete")

✅ Uploaded batch 1
✅ Uploaded batch 2
✅ Uploaded batch 3
✅ Uploaded batch 4
✅ Uploaded batch 5
✅ Upload complete
